# Weed Management LLM-Judge Evaluation

This notebook runs the synthetic farmer questions through the actual FastAPI app endpoint and uses a separate LLM model as a judge.

Evaluation flow:

1. Load synthetic farmer questions.
2. Call the real `/ask` FastAPI route using `TestClient`.
3. Match returned sources back to the source chunks.
4. Ask a separate judge model to score the answer.
5. Save local results.
6. Optionally log results to Braintrust when `BRAINTRUST_API_KEY` is available.

## 1. Setup

The app model is defined in `backend/rag.py`. The judge model is separate so the same model is not grading itself.

In [1]:
import json
import os
import sys
from pathlib import Path
from typing import List

from dotenv import load_dotenv
from fastapi.testclient import TestClient
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

DATA_DIR = PROJECT_ROOT / "data"
CHUNKS_PATH = DATA_DIR / "processed" / "chunks.json"
EVAL_DIR = DATA_DIR / "eval"
QUESTIONS_PATH = EVAL_DIR / "synthetic_farmer_questions.json"
RESULTS_JSON_PATH = EVAL_DIR / "llm_judge_results.json"
RESULTS_JSONL_PATH = EVAL_DIR / "llm_judge_results.jsonl"

EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Questions path:", QUESTIONS_PATH)
print("Chunks path:", CHUNKS_PATH)

Project root: /home/akaustiki/FDEhw/weed_chatbot
Questions path: /home/akaustiki/FDEhw/weed_chatbot/data/eval/synthetic_farmer_questions.json
Chunks path: /home/akaustiki/FDEhw/weed_chatbot/data/processed/chunks.json


## 2. Load Questions

The synthetic question file is a list of farmer-style question strings.

The judge context will come from the vector database by calling the same retrieval function used by the app, which retrieves the relevant chunks that were embedded from `data/processed/chunks.json`.


In [2]:
questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))

print(f"Loaded {len(questions)} questions")
print(f"Loaded {len(chunks)} source chunks")
print("First question:", questions[0])


Loaded 25 questions
Loaded 24 source chunks
First question: What are the key features to spot musk thistle quickly in my pasture?


## 3. Call the FastAPI App Endpoint

Use `TestClient` so evaluation goes through the same `/ask` endpoint that the frontend will use.

In [3]:
from backend.app import app

client = TestClient(app)


def ask_app(question: str):
    response = client.post("/ask", json={"question": question})
    response.raise_for_status()
    return response.json()

### Endpoint Smoke Tests

Check one answerable question and one out-of-scope question before running the full evaluation.

In [4]:
known_question = "When should herbicide be applied to musk thistle?"
known_result = ask_app(known_question)

print("QUESTION:", known_question)
print("ANSWER:", known_result["answer"])
print("SOURCES:", known_result["sources"])

QUESTION: When should herbicide be applied to musk thistle?
ANSWER: Herbicide should be applied to musk thistle during the spring rosette to early bolting stage or in the fall to rosettes. Specific herbicides and timings include: Aminopyralid (Milestone) at 5 fl. oz./acre in spring rosette to early bolting or fall to rosettes; Metsulfuron (Escort XP) at 1 oz. product/acre in spring from rosette through very early flower stage; and Chlorsulfuron (Telar) at 1 oz. product/acre in spring from rosette through early flower stage.
SOURCES: [{'source': 'NoxWeedMangementGuide2019LasAnimas.pdf', 'page': 2, 'column': 3, 'topic': 'Musk thistle'}]


In [5]:
out_of_scope_question = "What herbicide should I use for crabgrass in my lawn?"
out_of_scope_result = ask_app(out_of_scope_question)

print("QUESTION:", out_of_scope_question)
print("ANSWER:", out_of_scope_result["answer"])
print("SOURCES:", out_of_scope_result["sources"])

QUESTION: What herbicide should I use for crabgrass in my lawn?
ANSWER: The PDF does not provide enough information to answer this question.
SOURCES: []


## 4. Prepare Fresh Vector DB Context for the Judge

The app response already includes sources, but for judging we retrieve fresh context directly from ChromaDB using the same retrieval function as the backend.

This retrieval goes through the vector DB built from `chunks.json`, so the judge sees the relevant source chunks for the question. We still keep the app-returned sources in the saved result so citation behavior can be inspected separately.


In [6]:
from backend.rag import retrieve_documents, create_context


def format_sources(documents):
    return [
        {
            "source": doc.metadata["source"],
            "page": doc.metadata["page"],
            "column": doc.metadata["column"],
            "topic": doc.metadata["topic"],
        }
        for doc in documents
    ]


def build_judge_inputs(payload):
    documents = retrieve_documents(payload["question"], n_results=3)

    return {
        "question": payload["question"],
        "answer": payload["answer"],
        "sources": json.dumps(format_sources(documents), indent=2),
        "source_context": create_context(documents),
    }

## 5. Define the LLM Judge

The judge returns structured scores from 1 to 5.

Score meanings:

- **1**: poor or unsupported
- **3**: partially correct
- **5**: strong and well-supported

In [7]:
class JudgeResponse(BaseModel):
    retrieval_relevance: int = Field(ge=1, le=5, description="How relevant the returned sources are")
    groundedness: int = Field(ge=1, le=5, description="Whether the answer is supported by the source context")
    correctness: int = Field(ge=1, le=5, description="Whether the answer is factually correct according to the source context")
    completeness: int = Field(ge=1, le=5, description="Whether the answer covers the important details needed")
    citation_quality: int = Field(ge=1, le=5, description="Whether cited sources are appropriate and specific")
    pass_eval: bool = Field(description="True if the answer is acceptable overall")
    reason: str = Field(description="Brief explanation of the scores")


JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator for a PDF-grounded weed management chatbot.

Evaluate the chatbot answer using only the provided source context. Do not use outside knowledge.

Analyze the answer across these dimensions:

1. RETRIEVAL_RELEVANCE:
   Determine whether the retrieved source context is relevant to the farmer's question.
   - 5: The context directly answers the question.
   - 3: The context is partially related but incomplete.
   - 1: The context is unrelated.

2. GROUNDEDNESS:
   Determine whether the chatbot answer is supported by the source context.
   - 5: Every important claim is supported by the context.
   - 3: Some claims are supported, but some are vague or weakly supported.
   - 1: The answer includes unsupported or invented claims.

3. CORRECTNESS:
   Determine whether the answer is factually correct according to the source context.
   - 5: The answer is correct according to the context.
   - 3: The answer is partially correct but has minor errors or omissions.
   - 1: The answer contradicts the context.

4. COMPLETENESS:
   Determine whether the answer includes the important practical details available in the context.
   Consider identification clues, lifecycle, growth form, mechanical control, biological control,
   chemical control, herbicide rates, and herbicide timing when available.

5. CITATION_QUALITY:
   Determine whether the returned sources are specific and appropriate.
   - 5: Sources clearly support the answer.
   - 3: Sources are somewhat relevant but incomplete or too broad.
   - 1: Sources do not support the answer.

Special rule for out-of-scope questions:
If the source context does not answer the question and the chatbot says the PDF does not provide enough information,
score groundedness and correctness highly.

Your output MUST conform exactly to the JudgeResponse schema.
Do not include any text outside the structured output.
"""

JUDGE_USER_PROMPT = """
Evaluate this chatbot response.

<question>
{question}
</question>

<chatbot_answer>
{answer}
</chatbot_answer>

<returned_sources>
{sources}
</returned_sources>

<source_context>
{source_context}
</source_context>
"""

JUDGE_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", JUDGE_SYSTEM_PROMPT),
        ("human", JUDGE_USER_PROMPT),
    ]
)

judge_llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

judge_chain = (
    RunnableLambda(build_judge_inputs)
    | JUDGE_PROMPT
    | judge_llm.with_structured_output(JudgeResponse)
)

### Judge Smoke Test

Run one answer through the judge before evaluating the whole dataset.

In [8]:
judge_result = judge_chain.invoke(
    {
        "question": known_question,
        "answer": known_result["answer"],
    }
)

In [9]:
judge_result

JudgeResponse(retrieval_relevance=5, groundedness=5, correctness=5, completeness=5, citation_quality=5, pass_eval=True, reason='The chatbot answer is directly supported by the source context, which provides specific herbicide names, rates, and timing for musk thistle control. The answer accurately reflects the timing (spring rosette to early bolting or fall to rosettes) and includes the correct herbicide rates and stages as listed in the source. The cited source is specific to musk thistle and clearly supports the answer. All important practical details from the context are included.')

## 6. Run Evaluation Over All Synthetic Questions

This loop calls the app endpoint, judges the answer, and stores one result row per question.

In [10]:
eval_results = []

for index, question in enumerate(questions, start=1):
    print(f"[{index}/{len(questions)}] {question}")

    app_result = ask_app(question)

    judge_result = judge_chain.invoke(
        {
            "question": question,
            "answer": app_result["answer"],
        }
    )

    judge_scores = judge_result.model_dump()
    judge_reason = judge_scores.pop("reason")

    row = {
        "question": question,
        "answer": app_result["answer"],
        "app_sources": app_result["sources"],
        "judge_scores": judge_scores,
        "judge_reason": judge_reason,
    }

    eval_results.append(row)

print(f"Evaluated {len(eval_results)} questions")

[1/25] What are the key features to spot musk thistle quickly in my pasture?
[2/25] How can I tell musk thistle apart from other thistles when scouting fields?
[3/25] I've noticed bull thistle plants with thick stems and purple flowers late in the season. Is this normal for their lifecycle?
[4/25] The bull thistle in my field seems to be flowering earlier than usual this year. Could this indicate a change in its growth stages?
[5/25] What mechanical methods work best to control Canada thistle without damaging my crops?
[6/25] Can tilling at certain times reduce Canada thistle spread effectively on my farm?
[7/25] When is the best time to spray herbicide on diffuse knapweed for maximum effect?
[8/25] Should I wait until diffuse knapweed starts flowering before applying herbicides, or is earlier better?
[9/25] The salt cedar along my riverbank is growing thick and spreading fast. What control methods could I use to manage it effectively?
[10/25] I've seen salt cedar resprouting after cut

## 7. Save Local Results

Save both JSON and JSONL. JSON is easier to inspect, while JSONL is convenient for tooling.

In [11]:
RESULTS_JSON_PATH.write_text(
    json.dumps(eval_results, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

with RESULTS_JSONL_PATH.open("w", encoding="utf-8") as file:
    for row in eval_results:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved JSON: {RESULTS_JSON_PATH}")
print(f"Saved JSONL: {RESULTS_JSONL_PATH}")


Saved JSON: /home/akaustiki/FDEhw/weed_chatbot/data/eval/llm_judge_results.json
Saved JSONL: /home/akaustiki/FDEhw/weed_chatbot/data/eval/llm_judge_results.jsonl


## 8. Optional Braintrust Logging

If `BRAINTRUST_API_KEY` is present, this cell logs the local evaluation rows to Braintrust. If the key is missing, it skips logging and keeps the local files.

In [12]:
if os.getenv("BRAINTRUST_API_KEY"):
    from braintrust import init_logger, traced
    from braintrust.integrations.langchain import BraintrustCallbackHandler, set_global_handler

    braintrust_logger = init_logger(project="weed-management-chatbot")
    set_global_handler(BraintrustCallbackHandler())

    print("Braintrust tracing enabled.")
else:
    braintrust_logger = None
    print("BRAINTRUST_API_KEY not found. Braintrust tracing disabled.")

Braintrust tracing enabled.


In [13]:
@traced(name="Weed Management LLM Judge Evaluation")
def run_eval(questions):
    eval_results = []

    for index, question in enumerate(questions, start=1):
        print(f"[{index}/{len(questions)}] {question}")

        app_result = ask_app(question)

        judge_result = judge_chain.invoke(
            {
                "question": question,
                "answer": app_result["answer"],
            }
        )

        judge_scores = judge_result.model_dump()
        judge_reason = judge_scores.pop("reason")

        row = {
            "question": question,
            "answer": app_result["answer"],
            "app_sources": app_result["sources"],
            "judge_scores": judge_scores,
            "judge_reason": judge_reason,
        }

        eval_results.append(row)

    return eval_results


eval_results = run_eval(questions)

print(f"Evaluated {len(eval_results)} questions")

[1/25] What are the key features to spot musk thistle quickly in my pasture?
[2/25] How can I tell musk thistle apart from other thistles when scouting fields?
[3/25] I've noticed bull thistle plants with thick stems and purple flowers late in the season. Is this normal for their lifecycle?
[4/25] The bull thistle in my field seems to be flowering earlier than usual this year. Could this indicate a change in its growth stages?
[5/25] What mechanical methods work best to control Canada thistle without damaging my crops?
[6/25] Can tilling at certain times reduce Canada thistle spread effectively on my farm?
[7/25] When is the best time to spray herbicide on diffuse knapweed for maximum effect?
[8/25] Should I wait until diffuse knapweed starts flowering before applying herbicides, or is earlier better?
[9/25] The salt cedar along my riverbank is growing thick and spreading fast. What control methods could I use to manage it effectively?
[10/25] I've seen salt cedar resprouting after cut

## 9. Inspect Low-Scoring Results

Use this to manually review weak answers or retrieval failures.

In [14]:
def average_score(row):
    numeric_scores = [
        value
        for key, value in row["judge_scores"].items()
        if isinstance(value, int) and key != "pass_eval"
    ]
    return sum(numeric_scores) / len(numeric_scores)

low_scoring = sorted(eval_results, key=average_score)[:5]

for row in low_scoring:
    print("QUESTION:", row["question"])
    print("AVG SCORE:", round(average_score(row), 2))
    print("SCORES:", row["judge_scores"])
    print("REASON:", row["judge_reason"])
    print("ANSWER:", row["answer"][:800])
    print("-" * 100)

QUESTION: Can tilling at certain times reduce Canada thistle spread effectively on my farm?
AVG SCORE: 4.2
SCORES: {'retrieval_relevance': 3, 'groundedness': 5, 'correctness': 5, 'completeness': 5, 'citation_quality': 3, 'pass_eval': True}
REASON: The chatbot correctly states that the PDF does not provide enough information to answer the question about the timing of tillage for Canada thistle. The retrieved sources are partially relevant: they discuss Canada thistle control and mention mechanical methods like mowing, but do not specifically address tillage timing for Canada thistle. The answer is grounded and correct given the context. Completeness is high because the chatbot does not invent information and accurately reflects the limits of the source. Citation quality is moderate because the sources are related to weed control and Canada thistle, but do not specifically address the question. Overall, the answer is acceptable.
ANSWER: The PDF does not provide enough information to answ

## Observation

The LLM-judge evaluation shows that the weed management chatbot is working well for the current synthetic farmer questions. The app successfully calls the FastAPI `/ask` endpoint, retrieves relevant PDF chunks from ChromaDB, generates grounded answers, and returns specific source citations with page, column, and topic metadata.

The strongest results are for questions that directly match weed entries in the guide, such as Musk thistle, Canada thistle, Field bindweed, and Hoary cress. In these cases, the chatbot provides practical management details including identification clues, lifecycle, mechanical control, biological control, chemical control, herbicide rates, and timing when available in the source text.

The evaluation also confirms that the LLM judge can score answers across retrieval relevance, groundedness, correctness, completeness, and citation quality. Saving results to both JSON and JSONL makes the evaluation easy to inspect locally and reusable for future comparison. Optional Braintrust tracing can provide additional observability when an API key is available.

Overall, the current RAG pipeline appears reliable for the small PDF-based weed management dataset. Future improvements could focus on expanding the evaluation set, adding more out-of-scope questions, and comparing judge scores across prompt or retrieval changes.